In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from typing import List, Optional
from sqlalchemy import String, Integer, Date, ForeignKey, PrimaryKeyConstraint, text, create_engine, inspect
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy.orm import Session
from typing import Optional
from datetime import date
#cargar variables de conexion al servidor
load_dotenv()

usuario = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
puerto = os.getenv('DB_PORT')
nombre_db = os.getenv('DB_NAME')

engine = create_engine(f'postgresql://{usuario}:{password}@{host}/{nombre_db}')
# Prueba de conexión
try:
    with engine.connect() as connection:
        print("¡Conexión exitosa a PostgreSQL!")
except Exception as e:
    print(f"Error al conectar: {e}")

In [ ]:
#CAMPUS
query = '''
        CREATE TABLE IF NOT EXISTS campus  (
        id_campus SERIAL PRIMARY KEY,
        nombre_campus VARCHAR(25) NOT NULL
        );
        '''

with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#Restricciones de no duplicidad en datos
query = '''
ALTER TABLE campus
ADD CONSTRAINT unique_campus_nombre
UNIQUE (nombre_campus);
'''
with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#VERTICAL
query = '''
        CREATE TABLE IF NOT EXISTS vertical (
        id_vertical SERIAL PRIMARY KEY,
        nombre_vertical VARCHAR(25) NOT NULL
        );
        '''

with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#Restricciones de no duplicidad en datos
query = '''
ALTER TABLE vertical
ADD CONSTRAINT unique_vertical_nombre
UNIQUE (nombre_vertical);
'''
with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#PROYECTOS
query = '''
        CREATE TABLE IF NOT EXISTS proyectos (
        id_proyecto SERIAL PRIMARY KEY,
        nombre_proyecto VARCHAR(100) NOT NULL,
        id_vertical INT,
        FOREIGN KEY (id_vertical) REFERENCES vertical(id_vertical)
        );
        '''

with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#Restricciones de no duplicidad en datos
query = '''
ALTER TABLE proyectos
ADD CONSTRAINT unique_proyecto
UNIQUE (nombre_proyecto, id_vertical);
'''
with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#PROMOCION
query = '''
        CREATE TABLE IF NOT EXISTS promocion (
            id_promocion SERIAL PRIMARY KEY,
            nombre_mes VARCHAR(25) NOT NULL,
            fecha_comienzo DATE,
            id_campus INT,
            id_vertical INT,
            FOREIGN KEY (id_campus) REFERENCES campus(id_campus),
            FOREIGN KEY (id_vertical) REFERENCES vertical(id_vertical)
        );
        '''

with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#Restricciones de no duplicidad en datos
query = '''
ALTER TABLE promocion
ADD CONSTRAINT unique_promocion 
UNIQUE (nombre_mes, fecha_comienzo, id_campus, id_vertical);
'''
with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#ALUMNOS
query = '''
        CREATE TABLE IF NOT EXISTS alumnos (
            id_alumno SERIAL PRIMARY KEY,
            nombre VARCHAR(150) NOT NULL,
            email VARCHAR(150) NOT NULL,
            id_promocion INT,
            FOREIGN KEY (id_promocion) REFERENCES promocion(id_promocion)
        );
        '''

with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#Restricciones de no duplicidad en datos
query = '''
ALTER TABLE alumnos
ADD CONSTRAINT unique_alumno_email_promocion
UNIQUE (email, id_promocion);
'''
with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#PROFESORES
query = '''
        CREATE TABLE IF NOT EXISTS profesores (
            id_profesor SERIAL PRIMARY KEY,
            nombre_profesor VARCHAR(50) NOT NULL,
            rol VARCHAR(50) NOT NULL,
            modalidad VARCHAR(50) NOT NULL,
            id_promocion INT,
            FOREIGN KEY (id_promocion) REFERENCES promocion(id_promocion)
        );
        '''

with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#Restricciones de no duplicidad en datos
query = '''
ALTER TABLE profesores
ADD CONSTRAINT unique_profesor
UNIQUE (nombre_profesor, id_promocion);
'''
with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
#NOTAS
query = '''
        CREATE TABLE IF NOT EXISTS notas (
            id_alumno INT,
            id_proyecto INT,
            nota VARCHAR(50) NOT NULL,
            PRIMARY KEY (id_alumno, id_proyecto),
            FOREIGN KEY (id_alumno) REFERENCES alumnos(id_alumno),
            FOREIGN KEY (id_proyecto) REFERENCES proyectos(id_proyecto)
        );
        '''

with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
inspector = inspect(engine)
nombres_tablas = inspector.get_table_names()

for tabla in nombres_tablas:
    print(f"{tabla}")

In [ ]:
# insertar campus

with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO campus (nombre_campus) VALUES
            ('Madrid'),
            ('Valencia')
            ON CONFLICT (nombre_campus) DO NOTHING;
        """)
    )

In [ ]:
# insertar vertical

with engine.begin() as connection:
    connection.execute(
        text("""
            INSERT INTO vertical (nombre_vertical) VALUES
            ('Data Science'),
            ('Full Stack')
            ON CONFLICT (nombre_vertical) DO NOTHING;
        """)
    )

In [ ]:
query = """
            INSERT INTO promocion (nombre_mes, fecha_comienzo, id_campus, id_vertical) VALUES
            ('Septiembre', '2023-09-18', 1, 1),
            ('Febrero', '2024-02-12', 1, 1),
            ('Septiembre', '2023-09-18', 1, 2),
            ('Septiembre', '2024-12-09', 2, 2),
            ('Febrero', '2024-02-12', 2, 2)
            ON CONFLICT (nombre_mes, fecha_comienzo, id_campus, id_vertical) DO NOTHING;
        """

with engine.begin() as connection:
    connection.execute(
        text(query)
    )

In [ ]:
query = """
            INSERT INTO proyectos (nombre_proyecto, id_vertical) VALUES
            ('Proyecto_HLF', '1'),
            ('Proyecto_EDA', '1'),
            ('Proyecto_BBDD', '1'),
            ('Proyecto_ML', '1'),
            ('Proyecto_Deployment', '1'),
            ('Proyecto_WebDev', '2'),
            ('Proyecto_Frontend', '2'),
            ('Proyecto_Backend', '2'),
            ('Proyecto_React', '2'),
            ('Proyecto_FullStack', '2')
            ON CONFLICT (nombre_proyecto, id_vertical) DO NOTHING;
        """

with engine.begin() as connection:
    connection.execute(
        text(query)
        )

In [ ]:
query = """
            INSERT INTO alumnos (nombre, email, id_promocion) VALUES
            ('Jafet Casals', 'Jafet_Casals@gmail.com', 1),
            ('Jorge Manzanares', 'Jorge_Manzanares@gmail.com', 1),
            ('Onofre Adadia', 'Onofre_Adadia@gmail.com', 1),
            ('Merche Prada', 'Merche_Prada@gmail.com', 1),
            ('Pilar Abella', 'Pilar_Abella@gmail.com', 1),
            ('Leoncio Tena', 'Leoncio_Tena@gmail.com', 1),
            ('Odalys Torrijos', 'Odalys_Torrijos@gmail.com', 1),
            ('Eduardo Caparrós', 'Eduardo_Caparrós@gmail.com', 1),
            ('Ignacio Goicoechea', 'Ignacio_Goicoechea@gmail.com', 1),
            ('Clementina Santos', 'Clementina_Santos@gmail.com', 1),
            ('Daniela Falcó', 'Daniela_Falcó@gmail.com', 1),
            ('Abraham Vélez', 'Abraham_Vélez@gmail.com', 1),
            ('Maximiliano Menéndez', 'Maximiliano_Menéndez@gmail.com', 1),
            ('Anita Heredia', 'Anita_Heredia@gmail.com', 1),
            ('Eli Casas', 'Eli_Casas@gmail.com', 1),
            ('Guillermo Borrego', 'Guillermo_Borrego@gmail.com', 2),
            ('Sergio Aguirre', 'Sergio_Aguirre@gmail.com', 2),
            ('Carlito Carrión', 'Carlito_Carrión@gmail.com', 2),
            ('Haydée Figueroa', 'Haydée_Figueroa@gmail.com', 2),
            ('Chita Mancebo', 'Chita_Mancebo@gmail.com', 2),
            ('Joaquina Asensio', 'Joaquina_Asensio@gmail.com', 2),
            ('Cristian Sarabia', 'Cristian_Sarabia@gmail.com', 2),
            ('Isabel Ibáñez', 'Isabel_Ibáñez@gmail.com', 2),
            ('Desiderio Jordá', 'Desiderio_Jordá@gmail.com', 2),
            ('Rosalina Llanos', 'Rosalina_Llanos@gmail.com', 2),
            ('Amor Larrañaga', 'Amor_Larrañaga@gmail.com', 3),
            ('Teodoro Alberola', 'Teodoro_Alberola@gmail.com', 3),
            ('Cleto Plana', 'Cleto_Plana@gmail.com', 3),
            ('Aitana Sebastián', 'Aitana_Sebastián@gmail.com', 3),
            ('Dolores Valbuena', 'Dolores_Valbuena@gmail.com', 3),
            ('Julie Ferrer', 'Julie_Ferrer@gmail.com', 3),
            ('Mireia Cabañas', 'Mireia_Cabañas@gmail.com', 3),
            ('Flavia Amador', 'Flavia_Amador@gmail.com', 3),
            ('Albino Macias', 'Albino_Macias@gmail.com', 3),
            ('Ester Sánchez', 'Ester_Sánchez@gmail.com', 3),
            ('Luis Miguel Galvez', 'Luis_Miguel_Galvez@gmail.com', 3),
            ('Loida Arellano', 'Loida_Arellano@gmail.com', 3),
            ('Heraclio Duque', 'Heraclio_Duque@gmail.com', 3),
            ('Herberto Figueras', 'Herberto_Figueras@gmail.com', 3),
            ('Teresa Laguna', 'Teresa_Laguna@gmail.com', 4),
            ('Estrella Murillo', 'Estrella_Murillo@gmail.com', 4),
            ('Ernesto Uriarte', 'Ernesto_Uriarte@gmail.com', 4),
            ('Daniela Guitart', 'Daniela_Guitart@gmail.com', 4),
            ('Timoteo Trillo', 'Timoteo_Trillo@gmail.com', 4),
            ('Ricarda Tovar', 'Ricarda_Tovar@gmail.com', 4),
            ('Alejandra Vilaplana', 'Alejandra_Vilaplana@gmail.com', 4),
            ('Daniel Rosselló', 'Daniel_Rosselló@gmail.com', 4),
            ('Rita Olivares', 'Rita_Olivares@gmail.com', 4),
            ('Cleto Montes', 'Cleto_Montes@gmail.com', 4),
            ('Marino Castilla', 'Marino_Castilla@gmail.com', 4),
            ('Estefanía Valcárcel', 'Estefanía_Valcárcel@gmail.com', 4),
            ('Noemí Vilanova', 'Noemí_Vilanova@gmail.com', 4)
            ON CONFLICT (email, id_promocion) DO NOTHING;
            """

with engine.begin() as connection:
    connection.execute(
        text(query)
        )

In [ ]:
query = """
            INSERT INTO profesores (nombre_profesor, rol, modalidad, id_promocion) VALUES
            ('Noa Yáñez', 'TA', 'Presencial', 1),
            ('Saturnina Benitez', 'TA', 'Presencial', 1),
            ('Anna Feliu', 'TA', 'Presencial', 3),
            ('Rosalva Ayuso', 'TA', 'Presencial', 5),
            ('Ana Sofia Ferrer', 'TA', 'Presencial', 4),
            ('Angelica Corral', 'TA', 'Presencial', 2),
            ('Ariel Lledo', 'TA', 'Presencial', 1),
            ('Mario Prats', 'LI', 'Online', 4),
            ('Luis Angel Suarez', 'LI', 'Online', 3),
            ('Maria Dolores Diaz', 'LI', 'Online', 1)
            ON CONFLICT (nombre_profesor, id_promocion) DO NOTHING;
            """

with engine.begin() as connection:
    connection.execute(
        text(query)
        )

In [ ]:
# Leemos todos los CSV y los unimos en 1 dataframe.
csv = [ 'src/data/clase_1.csv','src/data/clase_2.csv','src/data/clase_3.csv','src/data/clase_4.csv']
lista_dfs = []
for i in csv:

    df_csv = pd.read_csv(i, sep=None, engine='python')
    df_csv.columns = df_csv.columns.str.strip()
    df_csv.drop(columns=["Email", "Promoción","Fecha_comienzo","Campus"], inplace=True, errors="ignore")
    df_csv_n = pd.melt(df_csv,id_vars=["Nombre"],var_name="Proyecto",value_name="Calificacion")
    lista_dfs.append(df_csv_n)

df_final = pd.concat(lista_dfs, ignore_index=True)

In [ ]:
# Buscamos las tablas alumnos y proyectos que lo vamos a necesitar
# alumnos
with engine.connect() as connection:
    query = text("SELECT id_alumno, nombre FROM alumnos;")
    resultado = connection.execute(query)
    df_alumnos = pd.DataFrame(resultado.all(), columns=resultado.keys())
    for fila in resultado:
        print(fila)
#proyectos
with engine.connect() as connection:
    query = text("SELECT id_proyecto, nombre_proyecto FROM proyectos;")
    resultado = connection.execute(query)
    df_proyectos = pd.DataFrame(resultado.all(), columns=resultado.keys())
    for fila in resultado:
        print(fila)

#vertical
with engine.connect() as connection:
    query = text("SELECT id_vertical, nombre_vertical FROM vertical;")
    resultado = connection.execute(query)
    df_vertical = pd.DataFrame(resultado.all(), columns=resultado.keys())
    for fila in resultado:
        print(fila)

#promocion
with engine.connect() as connection:
    query = text("SELECT id_promocion, nombre_mes, id_campus , id_vertical FROM promocion;")
    resultado = connection.execute(query)
    df_promocion = pd.DataFrame(resultado.all(), columns=resultado.keys())
    for fila in resultado:
        print(fila)     

In [ ]:
# merge y drop
df_final = df_final.merge(df_alumnos[['id_alumno', 'nombre']], left_on='Nombre', right_on='nombre', how='left')
df_final = df_final.drop(columns=['Nombre', 'nombre'])
#Ordenamos las columnas
cols = df_final.columns.tolist()
cols = cols[-1:] + cols[:-1]
df_final = df_final[cols]
#Cambio nombre de las columnas
df_final.columns = ['id_alumno', 'nombre_proyecto', 'nota']
# Arreglamos errores
df_final['nombre_proyecto'] = df_final['nombre_proyecto'].str.replace('FullSatck', 'FullStack', regex=False)
df_final['nombre_proyecto'] = df_final['nombre_proyecto'].str.replace('FrontEnd', 'Frontend', regex=False)
#merge con proyectos
df_final = df_final.merge(df_proyectos[['id_proyecto', 'nombre_proyecto']], on='nombre_proyecto', how='left')
#ordenamos
df_final = df_final[['id_alumno', 'id_proyecto', 'nota']]

In [ ]:
#Subimos los datos a la tabla NOTAS
df_final.to_sql('notas', con=engine, if_exists='append', index=False)

In [ ]:
df_profesores = pd.read_csv("src/data/claustro.csv", sep=None, engine='python')